# Watching the loop decide

This runs the orchestrator one stage at a time against a live question, so you
can see *where the decisions happen* rather than just the final answer.

Nothing here reimplements the project. Every cell imports the real modules and
calls them the way `coordinator.py` does. See
[README.md](README.md) for what it is and [DESIGN.md](DESIGN.md) for why it is
shaped this way.

One run costs about a dozen agent calls and a few minutes.

In [ ]:
import json, contextlib, io
from coordinator import classify_query, resolve_tickers, shortlist_for_scan
from agents.fundamentals_agent import run_fundamentals_agent
from agents.technical_agent import run_technical_agent
from agents.risk_critic_agent import run_risk_critic
from agents.portfolio_agent import run_portfolio_agent, partition_candidates

QUESTION = "Find me undervalued bank stocks"

def quiet(fn, *a, **k):
    "Run an agent without its progress chatter."
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

## 1. It reads the question before doing any work

This is the difference between a loop and a pipeline. A pipeline would already
be fetching data. This asks what kind of question it is first, and that answer
decides everything downstream.

In [ ]:
plan = quiet(classify_query, QUESTION)
plan

In [ ]:
tickers = resolve_tickers(plan)
tickers

## 2. Screen everything, cheaply

`sector_scan` means all five get the fundamentals agent and nothing else yet.
The expensive work is deliberately withheld until there is a reason to spend it.

The first one runs loudly so you can watch the agent choose its own tools. It
decides what to call and when it has enough — that sequence is not scripted
anywhere in my code.

In [ ]:
first = run_fundamentals_agent(tickers[0])

In [ ]:
screened = {tickers[0]: first}
for t in tickers[1:]:
    screened[t] = quiet(run_fundamentals_agent, t)

for t, v in screened.items():
    print(f"{t:6} {v['verdict']:15} score {v['valuation_score']}")

## 3. Rank them, don't test them

The original gate asked "is this undervalued?" and answered no for all eleven
stocks I tried it on, across two sectors. An absolute bar cannot answer a
comparative question.

Ranking always returns something, and the survivors carry a tag saying they
cleared on a *relative* basis so nothing downstream mistakes them for cheap.

In [ ]:
shortlist, dropped = shortlist_for_scan(screened)

print("through:")
for rank, (t, v) in enumerate(shortlist, 1):
    print(f"  {rank}. {t:6} {v['verdict']:15} score {v['valuation_score']}")
print("\nset aside:")
for t, v in dropped:
    print(f"     {t:6} {v['verdict']:15} score {v['valuation_score']}")

## 4. Now spend the expensive work

Technical and risk run only on the three that survived. The two that dropped
out never cost another call.

In [ ]:
results = {}
for t, v in dropped:
    results[t] = {"fundamentals": v, "skipped": "outside the top 3"}

for rank, (t, f_verdict) in enumerate(shortlist, 1):
    t_verdict = quiet(run_technical_agent, t)
    risk = quiet(run_risk_critic, f_verdict, t_verdict)
    results[t] = {"fundamentals": f_verdict, "technical": t_verdict, "risk": risk,
                  "screen": {"basis": "relative", "sector_rank": rank, "of": len(screened)}}
    print(f"{t:6} trend={t_verdict['trend_verdict']:14} entry={t_verdict['entry_quality']:15} risk={risk['verdict']}")

## 5. What the risk critic is allowed to see

It gets the other two agents' **conclusions** and nothing else. No reasoning,
no tool output, no working.

Show a critic how someone reached an answer and it tends to follow that path
instead of testing it. This is the same reason your own opinion drifts once
you have heard someone else's.

In [ ]:
sample = results[shortlist[0][0]]
print(json.dumps({"fundamentals": sample["fundamentals"],
                  "technical": sample["technical"]}, indent=2)[:900])

## 6. Vetoes are enforced in code, not requested in a prompt

Before the portfolio agent runs, rejected candidates are removed in Python.
A veto a later model could talk itself out of is not a veto.

In [ ]:
eligible, blocked = partition_candidates(results)
print("eligible:", list(eligible))
print("blocked: ", {t: r[:70] for t, r in blocked.items()})

## 7. Build the book

The portfolio agent is the only stage that sees more than one company at once,
which makes it the only one that can judge sizing and concentration at all.

It is told this shortlist came from a sector scan. Without that it reads its
own candidates as 100% concentration in one sector and declines to hold
anything — correct diversification logic, applied to a question that was never
about diversification.

In [ ]:
portfolio = quiet(run_portfolio_agent, results,
                  mandate={"query_type": plan["query_type"], "sector": plan.get("sector")})

for p in portfolio["positions"]:
    print(f"{p['ticker']:6} {p['allocation_pct']:>6.2f}%  {p['conviction']:8} {p['thesis'][:60]}")
print(f"{'CASH':6} {portfolio['cash_pct']:>6.2f}%")

## 8. What the code overrode

Every correction is recorded rather than applied quietly. A portfolio that got
silently rewritten is worse than one you can disagree with.

If `adjustments` is empty the model stayed inside the limits on its own. If it
is not, this is the model proposing and the code disposing.

In [ ]:
print("adjustments:")
for a in portfolio["adjustments"] or ["  (none - the model stayed inside the limits)"]:
    print(" ", a)

print("\nconcentration notes:")
for n in portfolio["concentration_notes"] or ["  (none)"]:
    print(" ", n[:110])

print("\nexcluded:")
for e in portfolio["excluded"]:
    print(f"  {e['ticker']:6} {e['reason'][:80]}")

---

Scores and verdicts move between runs, so re-running this will not reproduce
these numbers exactly. The routing, the funnel and the code-side limits behave
the same way every time.

Not investment advice. It is an architecture demo that happens to analyze
stocks.